<a href="https://colab.research.google.com/github/Mario-Cattaneo/SemesterProject/blob/main/Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Load Disnet splitting repository

In [1]:
import sys
import os

# 1. Clone the repository if it doesn't exist
if not os.path.exists('/content/DiSNet'):
    print("Cloning DiSNet repository...")
    !git clone https://github.com/ricsamikwa/DiSNet.git
else:
    print("DiSNet repository already exists.")

# 2. Change directory to the repository root
%cd /content/DiSNet

# 3. Force Python to prioritize local DiSNet files over system packages
# Inserting at index 0 ensures local files are found first
sys.path.insert(0, '/content/DiSNet')
sys.path.insert(0, '/content/DiSNet/main')

# 4. Patch model_vgg16.py to fix the super() reload bug
with open('models/model_vgg16.py', 'r') as file:
    content = file.read()
content = content.replace("super(VGG16, self).__init__()", "super().__init__()")
with open('models/model_vgg16.py', 'w') as file:
    file.write(content)
print("✓ Patched model_vgg16.py to prevent reload bugs.")

# 5. Patch model_convert.py to fix paths and torchvision imports
with open('main/model_convert.py', 'r') as file:
    content = file.read()
old_line = "Vgg_pretrined = torch.hub.load('pytorch/vision:v0.9.0', 'vgg16_bn', pretrained=True)"
new_line = "import torchvision.models as models\nVgg_pretrined = models.vgg16_bn(pretrained=True)"
content = content.replace(old_line, new_line)
old_path = "/home/eric/.cache/torch/hub/checkpoints/vgg16_bn-6c64b313.pth"
new_path = "/root/.cache/torch/hub/checkpoints/vgg16_bn-6c64b313.pth"
content = content.replace(old_path, new_path)
with open('main/model_convert.py', 'w') as file:
    file.write(content)
print("✓ Patched main/model_convert.py for Colab compatibility.")

# 6. Run the model conversion to generate modified weights
!mkdir -p main/models
!python3 main/model_convert.py

print("\n✓ Environment is fully configured and ready.")

DiSNet repository already exists.
/content/DiSNet
✓ Patched model_vgg16.py to prevent reload bugs.
✓ Patched main/model_convert.py for Colab compatibility.
conv_layers.0.0.weight : torch.Size([64, 3, 3, 3])
conv_layers.0.0.bias : torch.Size([64])
conv_layers.0.1.weight : torch.Size([64])
conv_layers.0.1.bias : torch.Size([64])
conv_layers.1.0.weight : torch.Size([64, 64, 3, 3])
conv_layers.1.0.bias : torch.Size([64])
conv_layers.1.1.weight : torch.Size([64])
conv_layers.1.1.bias : torch.Size([64])
conv_layers.3.0.weight : torch.Size([128, 64, 3, 3])
conv_layers.3.0.bias : torch.Size([128])
conv_layers.3.1.weight : torch.Size([128])
conv_layers.3.1.bias : torch.Size([128])
conv_layers.4.0.weight : torch.Size([128, 128, 3, 3])
conv_layers.4.0.bias : torch.Size([128])
conv_layers.4.1.weight : torch.Size([128])
conv_layers.4.1.bias : torch.Size([128])
conv_layers.6.0.weight : torch.Size([256, 128, 3, 3])
conv_layers.6.0.bias : torch.Size([256])
conv_layers.6.1.weight : torch.Size([256])
co

Specify devices and splitting configuration

In [2]:
%%writefile configurations.py
import numpy as np

# The IP address and port of the central edge server (sink node)
SERVER_ADDR = '192.168.1.38'
SERVER_PORT = 43000

# Total number of data samples in the dataset
N = 50000

# Model Configuration: Defines the layers of the VGG-16 model.
# Format: (Layer_Type, Input_Channels, Output_Channels, Kernel_Size, Output_Size, FLOPs)
# 'C' = Convolution, 'M' = MaxPool, 'D' = Dense (Fully Connected)
model_cfg = {
    'VGG' : [
        ('C', 3, 32, 3, 32*32*32, 32*32*32*3*3*3), ('M', 32, 32, 2, 32*16*16, 0),
        ('C', 32, 64, 3, 64*16*16, 64*16*16*3*3*32), ('M', 64, 64, 2, 64*8*8, 0),
        ('C', 64, 64, 3, 64*8*8, 64*8*8*3*3*64),
        ('D', 8*8*64, 128, 1, 64, 128*8*8*64),
        ('C', 64, 64, 3, 64*8*8, 64*8*8*3*3*64),
        ('D', 8*8*64, 128, 1, 64, 128*8*8*64),
        ('D', 128, 10, 1, 10, 128*10)
    ]
}

model_name = 'VGG'
model_size = 1.28 # Model size in MB
split_layer = [2, 3, 2]
model_len = 7

# The base processing rate of the devices (used to scale computation latency)
DEVICE_PACE_RATE = 10
rate = 10

# DAG Parallelization Limits: Defines the maximum number of parallel devices
# allowed to compute cooperatively at each of the 4 vertical stages.
max_par_partitions = [6, 5, 4, 3]

# Heterogeneous Hardware Profiles:
# trans_powers: Transmission power coefficients for each of the 3 device groups.
# comp_powers: Computation cost coefficients for each of the 3 device groups (lower = faster).
# device_power_groups: Defines the boundaries of the 3 device groups within the K=10 devices.
#   - Group 1 (Devices 0-2): Low-power (comp_power = 3.8)
#   - Group 2 (Devices 3-6): Medium-power (comp_power = 2.2)
#   - Group 3 (Devices 7-9): High-power/Server (comp_power = 1.2)
trans_powers = [1, 0.8, 1.1]
comp_powers = [3.8, 2.2, 1.2]
device_power_groups = [3, 7, 10]

# Vertical Stage Boundaries: Groups the 18 layers of VGG-16 into 4 vertical stages.
# Stage 1: Layers 0-4 | Stage 2: Layers 4-9 | Stage 3: Layers 9-14 | Stage 4: Layers 14-18
pos_max_par_partitions = [4, 9, 12, 18]
layer_range = np.array([[0, 4], [4, 9], [9, 14], [14, 18]])

# Training hyperparameters
R = 100
LR = 0.01
B = 100

# Total number of physical devices in the simulated mesh network
K = 10

# Number of local training iterations per device
iteration = {'192.168.1.33': 5, '192.168.1.41': 5, '192.168.1.40': 10, '192.168.1.42': 5, '192.168.1.43': 5}
random = True
random_seed = 0

Overwriting configurations.py


Perform splitting and extract the network and micro-splits

In [4]:
import torch
import torch.nn as nn
import numpy as np

# Ensure both root and main directories are in the Python path
sys.path.append(os.getcwd())
sys.path.append(os.path.join(os.getcwd(), 'main'))

from models.model_vgg16 import VGG16
from inference import get_partiton_info_DiSNet
from network import read_graph, select_path, determine_opt_neighbours, find_split_ratio, cap_trans_rate
from utils import create_partition_input
import configurations

class DAGEngine:
    """
    An explicit execution engine that handles spatial slicing, execution,
    and merging across the DAG of devices.
    """
    def __init__(self, model):
        self.model = model
        self.layer_range = configurations.layer_range
        self.kernel_sizes = [3,3,2,3,3,2,3,3,3,2,3,3,3,2,3,2,2,2,0,0,0]

    def execute_stage(self, in_tensor, stage_idx, partition_input, split_ratio, coding_scheme=None):
        start_layer = self.layer_range[stage_idx, 0]
        end_layer = self.layer_range[stage_idx, 1]
        if end_layer == 18:
            end_layer = 21 # Include fully connected layers

        layers_iter = end_layer - start_layer
        current_tensor = in_tensor

        for i in range(layers_iter):
            p = start_layer + i

            if p < 18:
                out_slices = []
                for j in range(len(partition_input[i][0])):
                    if partition_input[i][1][j] == 0:
                        break

                    # Spatial Slicing along Height (dimension 2)
                    start_idx = partition_input[i][0][j][0]
                    end_idx = partition_input[i][0][j][1] + 1
                    in_sub = current_tensor[:, :, start_idx:end_idx, :]

                    # Apply activation compression at the stage boundary
                    if coding_scheme is not None and i == 0:
                        in_sub = coding_scheme.compress(in_sub)

                    # Local Computation
                    output_sub = self.model([in_sub, p])

                    # Trim overlapping boundary pixels
                    if self.kernel_sizes[p] == 3:
                        if j not in [0, len(partition_input[i][0]) - 1]:
                            out_slices.append(output_sub[:, :, 1:-1, :])
                        elif j == 0:
                            out_slices.append(output_sub[:, :, :-1, :])
                        else:
                            out_slices.append(output_sub[:, :, 1:, :])
                    else:
                        out_slices.append(output_sub)

                # Merge spatial slices back together along Height
                current_tensor = out_slices[0]
                for k in range(len(out_slices) - 1):
                    current_tensor = torch.cat([current_tensor, out_slices[k+1]], dim=2)
            else:
                # Fully Connected Layers
                current_tensor = self.model([current_tensor, p])

        return current_tensor

# 1. Initialize the global model and engine
global_model = VGG16()
global_model_dict = global_model.state_dict()
global_model_dict.update(torch.load("main/vgg16-modify.pth"))
global_model.load_state_dict(global_model_dict)
global_model.eval()
if torch.cuda.is_available():
    global_model = global_model.to('cuda:0')

dag_engine = DAGEngine(global_model)

# 2. Extract and Print Network Topology and Device Stats
print("=" * 60)
print("               DISNET NETWORK EXTRACTION DETAILS               ")
print("=" * 60)

G = read_graph(0)
selected_path = select_path(G, 8, 4, 0.0)
print(f"Selected Routing Path (Nodes): {selected_path}")
print(f"Source Node: {selected_path[0]} | Sink Node: {selected_path[-1]}")
print("-" * 60)

print("Device Hardware Profiles on Path:")
for node in selected_path:
    # Determine power group
    group = 0
    for idx, boundary in enumerate(configurations.device_power_groups):
        if node < boundary:
            group = idx
            break
    comp_coef = configurations.comp_powers[group]
    trans_coef = configurations.trans_powers[group]
    print(f"  Node {node:2d} | Group: {group+1} | Comp Coef: {comp_coef:.1f} | Trans Coef: {trans_coef:.1f}")
print("-" * 60)

print("Vertical Stage Architecture:")
for idx, r in enumerate(configurations.layer_range):
    print(f"  Stage {idx+1}: Layers {r[0]:2d} to {r[1]:2d} | Max Parallel Neighbors: {configurations.max_par_partitions[idx]}")
print("=" * 60)

ImportError: cannot import name 'find_split_ratio' from 'network' (/content/DiSNet/network.py)

Coding scheme interface

In [ ]:
class BaseCodingScheme:
    """
    Unified interface for all task-oriented and standard coding schemes.
    """
    def __init__(self, name):
        self.name = name
        self.effective_bits = 32

    def adapt_to_constraints(self, target_latency, bandwidth, tensor_shape):
        """
        Dynamically adjusts compression parameters to satisfy the target latency
        under the given bandwidth.
        """
        raise NotImplementedError()

    def train_scheme(self, model, partition_input, par_trans_rate, comp_rate, split_ratio):
        """
        Optional offline training phase.
        """
        pass

    def compress(self, tensor):
        """
        Applies compression to the tensor.
        """
        raise NotImplementedError()

Basic quantization

In [ ]:
class UniformQuantization(BaseCodingScheme):
    """
    Standard Coding: Quantizes activations uniformly to a dynamically calculated bit-width.
    """
    def __init__(self):
        super().__init__(name="Uniform Quantization")
        self.bits = 32

    def adapt_to_constraints(self, target_latency, bandwidth, tensor_shape):
        num_elements = np.prod(tensor_shape)
        max_allowed_bits = (target_latency * (bandwidth * 1024 * 1024)) / num_elements
        self.bits = int(np.clip(max_allowed_bits, 2, 32))
        self.effective_bits = self.bits
        self.name = f"Quantization ({self.bits}-bit)"

    def compress(self, tensor):
        if self.effective_bits >= 32:
            return tensor

        min_val = tensor.min()
        max_val = tensor.max()
        scale = (max_val - min_val) / (2**self.effective_bits - 1)

        if scale == 0:
            return tensor

        quantized = torch.round((tensor - min_val) / scale)
        dequantized = quantized * scale + min_val
        return dequantized

Basic pruning

In [ ]:
class MagnitudePruning(BaseCodingScheme):
    """
    Standard Coding: Prunes activations based on magnitude to meet latency constraints.
    """
    def __init__(self):
        super().__init__(name="Magnitude Pruning")
        self.keep_ratio = 1.0

    def adapt_to_constraints(self, target_latency, bandwidth, tensor_shape):
        num_elements = np.prod(tensor_shape)
        max_allowed_bits = target_latency * (bandwidth * 1024 * 1024)
        self.keep_ratio = max_allowed_bits / (num_elements * 32)
        self.keep_ratio = float(np.clip(self.keep_ratio, 0.05, 1.0))
        self.effective_bits = 32 * self.keep_ratio
        self.name = f"Pruning ({int(self.keep_ratio*100)}% kept)"

    def compress(self, tensor):
        if self.keep_ratio >= 1.0:
            return tensor

        flat_tensor = tensor.abs().view(-1)
        k = int(self.keep_ratio * flat_tensor.numel())

        if k == 0:
            return torch.zeros_like(tensor)

        threshold, _ = torch.topk(flat_tensor, k)
        thresh_val = threshold[-1]

        mask = tensor.abs() >= thresh_val
        return tensor * mask

DAG-VDDBID

In [ ]:
import torch.nn.functional as F
import torch.optim as optim

class BottleneckNet(nn.Module):
    def __init__(self, in_features, bottleneck_features, bits, is_conv=True):
        super().__init__()
        self.is_conv = is_conv
        self.bits = bits

        if is_conv:
            self.encoder = nn.Conv2d(in_features, bottleneck_features, kernel_size=1, bias=False)
            self.decoder = nn.Conv2d(bottleneck_features, in_features, kernel_size=1, bias=False)
            nn.init.eye_(self.encoder.weight.squeeze(-1).squeeze(-1))
            nn.init.eye_(self.decoder.weight.squeeze(-1).squeeze(-1))
        else:
            self.encoder = nn.Linear(in_features, bottleneck_features, bias=False)
            self.decoder = nn.Linear(bottleneck_features, in_features, bias=False)
            nn.init.eye_(self.encoder.weight)
            nn.init.eye_(self.decoder.weight)

    def quantize(self, x):
        if self.bits >= 32:
            return x
        min_val = x.min()
        max_val = x.max()
        scale = (max_val - min_val) / (2**self.bits - 1)
        if scale == 0:
            return x
        quantized = torch.round((x - min_val) / scale)
        dequantized = quantized * scale + min_val
        return x + (dequantized - x).detach()

    def forward(self, x):
        latent = self.encoder(x)
        latent_q = self.quantize(latent)
        reconstructed = self.decoder(latent_q)
        return reconstructed


class DAG_VDDIB(BaseCodingScheme):
    """
    Our Proposed Strategy: Variational Distributed Deterministic Information Bottleneck.
    Dynamically adjusts both spatial downsampling and quantization to meet latency targets.
    """
    def __init__(self):
        super().__init__(name="DAG-VDDIB (Proposed)")
        self.downsample_factor = 1
        self.quantization_bits = 32
        self.nets = nn.ModuleDict()

    def adapt_to_constraints(self, target_latency, bandwidth, tensor_shape):
        num_elements = np.prod(tensor_shape)
        max_allowed_bits = target_latency * (bandwidth * 1024 * 1024)
        target_effective_bits = max_allowed_bits / num_elements

        best_f, best_n = 1, 32
        min_error = float('inf')

        for f in [1, 2, 3, 4]:
            for n in [2, 4, 8, 16, 32]:
                eff_bits = n / (f ** 2)
                if eff_bits <= target_effective_bits:
                    error = target_effective_bits - eff_bits
                    if error < min_error:
                        min_error = error
                        best_f, best_n = f, n

        self.downsample_factor = best_f
        self.quantization_bits = best_n
        self.effective_bits = best_n / (best_f ** 2)
        self.name = f"DAG-VDDIB (Down={best_f}x, {best_n}-bit)"

    def train_scheme(self, model, partition_input, par_trans_rate, comp_rate, split_ratio):
        print(f"  Training bottleneck layers offline for {self.name}...")

        collected_activations = []
        x_input = torch.randn(10, 3, 224, 224)
        if torch.cuda.is_available():
            x_input = x_input.to('cuda:0')
            model.to('cuda:0')

        layer_range = configurations.layer_range

        with torch.no_grad():
            for m in range(x_input.shape[0]):
                output = x_input[m:m+1]
                stage_activations = []
                for j in range(len(layer_range)):
                    output = dag_engine.execute_stage(output, j, partition_input[layer_range[j,0]:layer_range[j,1]], split_ratio[j])
                    stage_activations.append(output.clone())
                collected_activations.append(stage_activations)

        for j in range(len(layer_range)):
            _ = self.compress(collected_activations[0][j])

        optimizer = optim.Adam(self.nets.parameters(), lr=0.01)
        criterion = nn.MSELoss()

        for step in range(40):
            optimizer.zero_grad()
            loss = 0
            for m in range(len(collected_activations)):
                for j in range(len(layer_range)):
                    original_act = collected_activations[m][j]
                    reconstructed = self.compress(original_act)
                    loss += criterion(reconstructed, original_act)
            loss.backward()
            optimizer.step()

        print("  Training Complete.")

    def compress(self, tensor):
        is_conv = (tensor.dim() == 4)
        in_features = tensor.shape[1]
        key = f"{in_features}_{'conv' if is_conv else 'linear'}"

        if key not in self.nets:
            net = BottleneckNet(in_features, in_features, self.quantization_bits, is_conv=is_conv)
            net = net.to(tensor.device)
            self.nets[key] = net

        if is_conv and self.downsample_factor > 1:
            orig_size = (tensor.shape[2], tensor.shape[3])
            new_size = (max(1, orig_size[0] // self.downsample_factor),
                        max(1, orig_size[1] // self.downsample_factor))

            latent = F.interpolate(tensor, size=new_size, mode='bilinear', align_corners=False)
            latent_q = self.nets[key].quantize(latent)
            reconstructed = F.interpolate(latent_q, size=orig_size, mode='bilinear', align_corners=False)
            return reconstructed
        else:
            return self.nets[key](tensor)

Evaluation

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

def run_evaluation_simulation(model, coding_scheme, target_latency, bandwidth, partition_input, par_trans_rate, comp_rate, split_ratio):
    filename = "input/dog.jpg"
    input_image = Image.open(filename)
    preprocess = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    input_batch = preprocess(input_image).unsqueeze(0)
    if torch.cuda.is_available():
        input_batch = input_batch.to('cuda:0')

    layer_range = configurations.layer_range
    output = input_batch

    with torch.no_grad():
        first_stage_out = dag_engine.execute_stage(output, 0, partition_input[layer_range[0,0]:layer_range[0,1]], split_ratio[0])

    coding_scheme.adapt_to_constraints(target_latency, bandwidth, first_stage_out.shape)
    coding_scheme.train_scheme(model, partition_input, par_trans_rate, comp_rate, split_ratio)

    with torch.no_grad():
        for j in range(len(layer_range)):
            output = dag_engine.execute_stage(output, j, partition_input[layer_range[j,0]:layer_range[j,1]], split_ratio[j], coding_scheme)

        probabilities = torch.nn.functional.softmax(output[0], dim=0)
        top5_prob, _ = torch.topk(probabilities, 5)
        accuracy = sum(top5_prob).item()

    return accuracy

# --- Execution Loop ---

fixed_bandwidth = 5.0  # Mbps
target_latencies = [0.05, 0.1, 0.2, 0.5, 1.0]  # seconds

# Extract the actual simulation parameters from the graph
G = read_graph(0)
selected_path = select_path(G, 8, 4, 0.0)
current_point_on_path = 0
split_ratio = []
devices = []
trans_rate_forward = []
par_trans_rate = []

for i in range(len(configurations.max_par_partitions)):
    current_max_par_partitions = configurations.max_par_partitions[i]
    p = determine_opt_neighbours(G, selected_path, current_max_par_partitions, current_point_on_path, 0.0)
    current_point_on_path = selected_path.index(p)
    current_neighbours = []
    for n in G.neighbors(p):
        current_neighbours.append([n, G.nodes[n]['weight'], G[p][n]['weight']])
    if current_point_on_path >= len(selected_path)-1:
        current_neighbours.append([p, G.nodes[p]['weight'], G[p][selected_path[current_point_on_path-1]]['weight']])
    else:
        current_neighbours.append([p, G.nodes[p]['weight'], G[p][selected_path[current_point_on_path+1]]['weight']])

    horizontal_split_ratio, device, in_throughput = find_split_ratio(current_neighbours)
    split_ratio.append(horizontal_split_ratio)
    par_trans_rate.append(in_throughput)

par_trans_rate = cap_trans_rate(fixed_bandwidth, par_trans_rate)
partition_input = create_partition_input(split_ratio)
comp_rate = split_ratio

# Define the schemes to evaluate
schemes = [
    UniformQuantization(),
    MagnitudePruning(),
    DAG_VDDIB()
]

all_results = {}

for scheme in schemes:
    accuracies = []
    print(f"\nEvaluating Scheme: {scheme.name}")
    for target_lat in target_latencies:
        acc = run_evaluation_simulation(
            model=global_model,
            coding_scheme=scheme,
            target_latency=target_lat,
            bandwidth=fixed_bandwidth,
            partition_input=partition_input,
            par_trans_rate=par_trans_rate,
            comp_rate=comp_rate,
            split_ratio=split_ratio
        )
        accuracies.append(acc)
        print(f"  Target Latency: {target_lat:.2f}s | Adapted to: {scheme.name} | Accuracy: {acc:.4f}")
    all_results[scheme.name] = accuracies
    print("-" * 60)

# Plot the results
plt.figure(figsize=(10, 6))
for name, accs in all_results.items():
    plt.plot(target_latencies, accs, marker='o', linestyle='-', label=name)

plt.title(f"Accuracy vs. Target Latency (Fixed Bandwidth: {fixed_bandwidth} Mbps)")
plt.xlabel("Target Latency Constraint (seconds)")
plt.ylabel("Top-5 Accuracy")
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()